# Predicting Student Health Risk — v7: Pseudo-Labeling
### Kaggle Playground Series S6E7

**Recap:**
- v3 LightGBM (tuned): CV 0.94965, leaderboard **0.95011** (current best)
- v4 CatBoost: CV 0.94902
- v5 Ensemble: CV 0.94961 (no real gain)
- v6 Feature engineering: CV 0.94966 (no real gain)

**What pseudo-labeling does:** since your test set is large (295,753 rows) and your model is
already fairly confident on most of it, we can treat the model's *most confident* test
predictions as extra "labeled" training data, retrain including them, and see if the extra
volume helps the model generalize better — even though those extra labels aren't ground truth,
just the model's own high-confidence guesses.

**Why this needs a careful setup (not just "add everything and retrain"):**
- If we validate pseudo-labeling using the same rows we generated pseudo-labels from, we'd be
  fooling ourselves — of course a model does well on data it labeled itself.
- So this notebook uses a clean **held-out validation split from the original training data**
  that is NEVER used to generate pseudo-labels or in the pseudo-labeled training — only to
  honestly check afterward whether pseudo-labeling actually helped.

**Steps:**
1. Split original train into `train_main` (80%) and `val_holdout` (20%) — `val_holdout` has real
   ground-truth labels and stays untouched until the final comparison.
2. Train a baseline model on `train_main` only, score it on `val_holdout` — this is our honest
   "no pseudo-labeling" reference score.
3. Use that same model to predict on the actual **test set**, and keep only the most confident
   predictions (probability above a threshold) as pseudo-labels.
4. Retrain a new model on `train_main` + pseudo-labeled test rows.
5. Score this new model on the same untouched `val_holdout` — compare directly against step 2.
6. If it genuinely helps, retrain one final time on ALL original train data + pseudo-labels, and
   predict on test for submission.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/playground-series-s6e7"


## 1. Load and preprocess (same as v3)

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

target_col = "health_condition"
feature_cols = [c for c in train.columns if c not in [target_col, "id"]]

X_full = train[feature_cols].copy()
y_full = train[target_col].copy()
X_test = test[feature_cols].copy()

cat_cols = X_full.select_dtypes(include="object").columns.tolist()
num_cols = X_full.select_dtypes(exclude="object").columns.tolist()

for c in num_cols:
    med = X_full[c].median()
    X_full[c] = X_full[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

for c in cat_cols:
    X_full[c] = X_full[c].fillna("missing")
    X_test[c] = X_test[c].fillna("missing")
    all_cats = pd.concat([X_full[c], X_test[c]]).astype(str).unique()
    X_full[c] = X_full[c].astype(str).astype(pd.CategoricalDtype(categories=all_cats))
    X_test[c] = X_test[c].astype(str).astype(pd.CategoricalDtype(categories=all_cats))

target_encoder = LabelEncoder()
y_full_enc = target_encoder.fit_transform(y_full)
print("Full train shape:", X_full.shape, " Test shape:", X_test.shape)
print(dict(zip(target_encoder.classes_, range(len(target_encoder.classes_)))))

# Best params from v3's Optuna search
best_params = {
    'n_estimators': 500,
    'learning_rate': 0.03426069502422001,
    'num_leaves': 23,
    'max_depth': 9,
    'min_child_samples': 18,
    'subsample': 0.7293691429332224,
    'colsample_bytree': 0.5962321115660644,
    'reg_alpha': 5.178964936328794e-07,
    'reg_lambda': 7.656695372261228e-07,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

Full train shape: (690088, 13)  Test shape: (295753, 13)
{'at-risk': 0, 'fit': 1, 'unhealthy': 2}


## 2. Held-out validation split (never touched by pseudo-labeling)

This 20% split is our honest referee. Everything we do with pseudo-labels gets checked against
this at the end — it stays completely separate from the pseudo-labeling process.

In [3]:
X_train_main, X_val_holdout, y_train_main, y_val_holdout = train_test_split(
    X_full, y_full_enc, test_size=0.2, stratify=y_full_enc, random_state=42
)
print("train_main:", X_train_main.shape, " val_holdout:", X_val_holdout.shape)

train_main: (552070, 13)  val_holdout: (138018, 13)


## 3. Baseline: train on train_main only, score on val_holdout

This is our honest "no pseudo-labeling" reference score — note this will be somewhat lower than
v3's full-data CV score (0.94965), since this model only saw 80% of the training data instead of
using all of it via 5-fold CV. That's expected and fine; the comparison that matters is
baseline-on-80%-data vs pseudo-labeled-on-80%-data+extra, both scored on the same val_holdout.

In [4]:
baseline_model = LGBMClassifier(**best_params)
baseline_model.fit(
    X_train_main, y_train_main,
    eval_set=[(X_val_holdout, y_val_holdout)],
    callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
)

baseline_val_pred = baseline_model.predict(X_val_holdout)
baseline_score = balanced_accuracy_score(y_val_holdout, baseline_val_pred)
print(f"Baseline (train_main only) balanced accuracy on val_holdout: {baseline_score:.5f}")

Baseline (train_main only) balanced accuracy on val_holdout: 0.95012


## 4. Generate pseudo-labels from confident test predictions

We use the SAME baseline model (trained only on train_main) to predict on the actual test set,
then keep only rows where the model is highly confident (max predicted probability above
`CONFIDENCE_THRESHOLD`). Higher threshold = fewer but more trustworthy pseudo-labels.

In [5]:
CONFIDENCE_THRESHOLD = 0.95

test_proba = baseline_model.predict_proba(X_test)
test_pred_enc = np.argmax(test_proba, axis=1)
test_confidence = np.max(test_proba, axis=1)

confident_mask = test_confidence >= CONFIDENCE_THRESHOLD
n_confident = confident_mask.sum()
print(f"Confident test rows (>= {CONFIDENCE_THRESHOLD} probability): {n_confident} / {len(X_test)} ({100*n_confident/len(X_test):.1f}%)")

# Check the class distribution of the pseudo-labels — watch for it skewing even more
# heavily toward the majority class than the real data does
pseudo_labels_preview = target_encoder.inverse_transform(test_pred_enc[confident_mask])
print(pd.Series(pseudo_labels_preview).value_counts())

Confident test rows (>= 0.95 probability): 124779 / 295753 (42.2%)
at-risk      92106
unhealthy    18784
fit          13889
Name: count, dtype: int64


## 5. Retrain with train_main + confident pseudo-labels, score on val_holdout

This is the real test: does adding pseudo-labeled data actually improve generalization, measured
on data the model has never seen or influenced in any way?

In [6]:
X_pseudo = X_test[confident_mask].copy()
y_pseudo = test_pred_enc[confident_mask]

X_train_plus_pseudo = pd.concat([X_train_main, X_pseudo], axis=0, ignore_index=True)
y_train_plus_pseudo = np.concatenate([y_train_main, y_pseudo])

print(f"train_main size: {len(X_train_main)}  ->  train_main + pseudo size: {len(X_train_plus_pseudo)}")

pseudo_model = LGBMClassifier(**best_params)
pseudo_model.fit(
    X_train_plus_pseudo, y_train_plus_pseudo,
    eval_set=[(X_val_holdout, y_val_holdout)],
    callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
)

pseudo_val_pred = pseudo_model.predict(X_val_holdout)
pseudo_score = balanced_accuracy_score(y_val_holdout, pseudo_val_pred)

print(f"\nBaseline (train_main only):        {baseline_score:.5f}")
print(f"With pseudo-labels (train_main+PL): {pseudo_score:.5f}")
print(f"Difference: {pseudo_score - baseline_score:+.5f}")

train_main size: 552070  ->  train_main + pseudo size: 676849

Baseline (train_main only):        0.95012
With pseudo-labels (train_main+PL): 0.94991
Difference: -0.00021


## 6. Interpret the result before proceeding

- If `pseudo_score > baseline_score` by a meaningful margin (more than the noise you've seen
  elsewhere, roughly +0.001 or more), pseudo-labeling is genuinely helping — proceed to step 7.
- If the difference is tiny or negative, pseudo-labeling isn't adding value here — don't submit
  this approach; stick with your v3 result instead.
- You can also try adjusting `CONFIDENCE_THRESHOLD` (e.g. 0.9 or 0.99) and re-running steps 4-5
  to see if a different confidence cutoff changes the picture.

## 7. If it helped: retrain on ALL original data + pseudo-labels, predict test, save submission

Only run this section if step 5 showed a real improvement.

In [7]:
final_model = LGBMClassifier(**best_params)

X_final_train = pd.concat([X_full, X_pseudo], axis=0, ignore_index=True)
y_final_train = np.concatenate([y_full_enc, y_pseudo])

# Use a small internal validation split just for early stopping (not for the final decision)
X_ft, X_fv, y_ft, y_fv = train_test_split(
    X_final_train, y_final_train, test_size=0.1, stratify=y_final_train, random_state=42
)

final_model.fit(
    X_ft, y_ft,
    eval_set=[(X_fv, y_fv)],
    callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
)

final_test_proba = final_model.predict_proba(X_test)
final_preds_enc = np.argmax(final_test_proba, axis=1)
final_preds = target_encoder.inverse_transform(final_preds_enc)

submission = pd.DataFrame({"id": test["id"], "health_condition": final_preds})
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
submission.to_csv("submission.csv", index=False)

print(pd.Series(final_preds).value_counts())
submission.head()

at-risk      240014
unhealthy     33850
fit           21889
Name: count, dtype: int64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## Next steps

1. Check the step 5 comparison first — this is the honest signal, don't skip straight to
   submitting.
2. If it helped, try 1-2 more rounds of the same idea (retrain, generate new pseudo-labels from
   the improved model, repeat) — but diminishing returns usually set in fast, 1-2 rounds is
   typically enough.
3. If it didn't help, that's a valid and useful finding too — it means your model is already
   about as good as it can get from more (self-generated) data; real gains would need either
   genuinely new information (external data, if allowed) or a different modeling approach.
4. Whatever the outcome, you'll have learned a real technique used in many top Kaggle solutions
   — pseudo-labeling is one of the most common tricks in Playground-style competitions
   specifically because test sets are large and public.

Keep versioning: this is v7 (pseudo-labeling on tuned LightGBM).